# EasyOCR Test — CNIE OCR

**This notebook tests EasyOCR in multiple modes:**
1. **RAW mode** — No preprocessing, no cropping, just feed the whole image
2. **Field cropping mode** — Crop specific regions

**EasyOCR Note:** Arabic can only combine with English, not French.
So we use two readers:
- Arabic + English reader
- French reader

In [ ]:
# Run ONCE to install EasyOCR
!pip install easyocr

In [ ]:
import easyocr
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"EasyOCR version: {easyocr.__version__}")

---
## Initialize EasyOCR Readers

Two readers due to language limitations:
- **Arabic + English** — for Arabic text and numbers/dates
- **French** — for French text

In [ ]:
# Initialize readers (models download on first run)

print("Loading Arabic + English reader...")
t0 = time.time()
reader_ar_en = easyocr.Reader(['ar', 'en'], gpu=False, verbose=False)
print(f"Arabic+English reader ready ({time.time()-t0:.1f}s)")

print("\nLoading French reader...")
t0 = time.time()
reader_fr = easyocr.Reader(['fr'], gpu=False, verbose=False)
print(f"French reader ready ({time.time()-t0:.1f}s)")

print("\nReaders loaded. Ready for OCR.")

---
# SECTION 1: RAW OCR TEST

**No preprocessing. No region cropping. Just the raw image.**

In [ ]:
# Load the RAW front image
raw_front = None
raw_filename = None

for fname in ['front.jpeg', 'front.jpg', 'front.png', 'front_cropped.jpg']:
    raw_front = cv2.imread(fname)
    if raw_front is not None:
        raw_filename = fname
        break

if raw_front is None:
    print("ERROR: Could not load any front image")
    print("Place your CIN image as 'front.jpeg' or 'front_cropped.jpg'")
else:
    # Auto-rotate if needed
    h, w = raw_front.shape[:2]
    if h > w * 1.2:
        raw_front = cv2.rotate(raw_front, cv2.ROTATE_90_COUNTERCLOCKWISE)
        print("Auto-rotated to landscape")
    
    print(f"Loaded: {raw_filename}")
    print(f"Size: {raw_front.shape[1]}w x {raw_front.shape[0]}h")
    
    plt.figure(figsize=(14, 9))
    plt.imshow(cv2.cvtColor(raw_front, cv2.COLOR_BGR2RGB))
    plt.title(f'RAW Image — {raw_front.shape[1]}x{raw_front.shape[0]}')
    plt.axis('off')
    plt.show()

In [ ]:
# RUN RAW OCR with Arabic + English reader

if raw_front is not None:
    print("Running EasyOCR (Arabic + English) on RAW image...")
    print("No preprocessing. No cropping.")
    print("=" * 70)
    
    t0 = time.time()
    results_ar_en = reader_ar_en.readtext(raw_front, detail=1, paragraph=False)
    elapsed = time.time() - t0
    
    print(f"\nOCR completed in {elapsed:.2f}s")
    print(f"Detected {len(results_ar_en)} text regions")
    print("=" * 70)
    
    print("\nALL DETECTED TEXT (Arabic + English):")
    print("-" * 70)
    for i, item in enumerate(results_ar_en):
        bbox, text, conf = item
        print(f"{i+1:2d}. [{conf:.2f}] {text}")
else:
    print("No image loaded.")

In [ ]:
# RUN RAW OCR with French reader

if raw_front is not None:
    print("Running EasyOCR (French) on RAW image...")
    print("=" * 70)
    
    t0 = time.time()
    results_fr = reader_fr.readtext(raw_front, detail=1, paragraph=False)
    elapsed = time.time() - t0
    
    print(f"\nOCR completed in {elapsed:.2f}s")
    print(f"Detected {len(results_fr)} text regions")
    print("=" * 70)
    
    print("\nALL DETECTED TEXT (French):")
    print("-" * 70)
    for i, item in enumerate(results_fr):
        bbox, text, conf = item
        print(f"{i+1:2d}. [{conf:.2f}] {text}")
else:
    print("No image loaded.")

In [ ]:
# VISUALIZE — Draw bounding boxes (Arabic + English results)

if raw_front is not None and results_ar_en:
    img_boxes = raw_front.copy()
    
    for i, item in enumerate(results_ar_en):
        bbox, text, conf = item
        pts = np.array(bbox, dtype=np.int32)
        
        # Color by confidence
        if conf > 0.8:
            color = (0, 255, 0)   # Green
        elif conf > 0.5:
            color = (0, 255, 255) # Yellow
        else:
            color = (0, 0, 255)   # Red
        
        cv2.polylines(img_boxes, [pts], True, color, 2)
        cv2.putText(img_boxes, str(i+1), tuple(pts[0]), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(img_boxes, cv2.COLOR_BGR2RGB))
    plt.title(f'RAW OCR — {len(results_ar_en)} detections (Green=high, Yellow=medium, Red=low)')
    plt.axis('off')
    plt.show()

In [ ]:
# STATISTICS

if raw_front is not None and results_ar_en:
    confs = [item[2] for item in results_ar_en]
    
    print("=" * 50)
    print("STATISTICS (Arabic + English reader)")
    print("=" * 50)
    print(f"Total detections: {len(results_ar_en)}")
    print(f"Avg confidence:   {np.mean(confs):.2f}")
    print(f"Min confidence:   {np.min(confs):.2f}")
    print(f"Max confidence:   {np.max(confs):.2f}")
    print(f"High conf (>0.8): {sum(1 for c in confs if c > 0.8)}")
    print(f"Low conf (<0.5):  {sum(1 for c in confs if c < 0.5)}")

---
# SECTION 2: RAW OCR on BACK SIDE

In [ ]:
# Load back image
raw_back = None

for fname in ['back.jpeg', 'back.jpg', 'back.png']:
    raw_back = cv2.imread(fname)
    if raw_back is not None:
        break

if raw_back is None:
    print("No back image found. Skipping.")
else:
    h, w = raw_back.shape[:2]
    if h > w * 1.2:
        raw_back = cv2.rotate(raw_back, cv2.ROTATE_90_COUNTERCLOCKWISE)
    
    print(f"Back loaded: {raw_back.shape[1]}w x {raw_back.shape[0]}h")
    
    plt.figure(figsize=(14, 9))
    plt.imshow(cv2.cvtColor(raw_back, cv2.COLOR_BGR2RGB))
    plt.title('RAW Back Image')
    plt.axis('off')
    plt.show()

In [ ]:
# RAW OCR on back

if raw_back is not None:
    print("Running EasyOCR on back image...")
    
    t0 = time.time()
    back_results = reader_ar_en.readtext(raw_back, detail=1, paragraph=False)
    elapsed = time.time() - t0
    
    print(f"Detected {len(back_results)} regions in {elapsed:.2f}s")
    print("-" * 50)
    
    for i, item in enumerate(back_results):
        bbox, text, conf = item
        print(f"{i+1:2d}. [{conf:.2f}] {text}")
    
    # Visualize
    img_boxes = raw_back.copy()
    for i, item in enumerate(back_results):
        bbox, text, conf = item
        pts = np.array(bbox, dtype=np.int32)
        color = (0, 255, 0) if conf > 0.8 else (0, 255, 255) if conf > 0.5 else (0, 0, 255)
        cv2.polylines(img_boxes, [pts], True, color, 2)
        cv2.putText(img_boxes, str(i+1), tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(img_boxes, cv2.COLOR_BGR2RGB))
    plt.title(f'Back OCR — {len(back_results)} detections')
    plt.axis('off')
    plt.show()

---
# SECTION 3: FIELD-BASED OCR (with cropping)

Test with predefined field regions for comparison.

In [ ]:
# Load cropped card
front = cv2.imread('front_cropped.jpg')

if front is None:
    print("front_cropped.jpg not found.")
    print("Run Tesseract notebook first to generate it.")
else:
    print(f'Cropped card: {front.shape[1]}w x {front.shape[0]}h')
    
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
    plt.title('Cropped Card')
    plt.axis('off')
    plt.show()

In [ ]:
# Field definitions
PADDING = 10

FRONT_FIELDS = {
    "first_name_fr":      {"x": 0,   "y": 148, "w": 500, "h": 52,  "lang": "fr"},
    "last_name_fr":       {"x": 0,   "y": 218, "w": 500, "h": 50,  "lang": "fr"},
    "date_of_birth":      {"x": 160, "y": 255, "w": 210, "h": 54,  "lang": "en"},
    "place_of_birth_fr":  {"x": 0,   "y": 322, "w": 500, "h": 50,  "lang": "fr"},
    "expiry_date":        {"x": 200, "y": 365, "w": 200, "h": 45,  "lang": "en"},
    "first_name_ar":      {"x": 330, "y": 130, "w": 270, "h": 48,  "lang": "ar"},
    "last_name_ar":       {"x": 330, "y": 204, "w": 270, "h": 48,  "lang": "ar"},
    "card_number":        {"x": 590, "y": 405, "w": 220, "h": 42,  "lang": "en"},
    "gender":             {"x": 800, "y": 400, "w": 56,  "h": 45,  "lang": "en"},
}

# Map to readers
READERS = {
    "ar": reader_ar_en,  # Arabic uses ar+en reader
    "en": reader_ar_en,  # English uses ar+en reader
    "fr": reader_fr,     # French uses fr reader
}

print(f"Defined {len(FRONT_FIELDS)} fields.")

In [ ]:
# Test all fields

def ocr_field(img, x, y, w, h, field_name, lang, padding=PADDING):
    ih, iw = img.shape[:2]
    x1, y1 = max(0, x - padding), max(0, y - padding)
    x2, y2 = min(iw, x + w + padding), min(ih, y + h + padding)
    
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    
    reader = READERS[lang]
    t0 = time.time()
    result = reader.readtext(crop, detail=1, paragraph=False)
    elapsed = time.time() - t0
    
    text = " ".join([r[1] for r in result]) if result else ""
    conf = min([r[2] for r in result]) if result else 0.0
    
    plt.figure(figsize=(10, 1.5))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(f"{field_name} | '{text}' | conf={conf:.2f} | {elapsed*1000:.0f}ms", fontsize=10)
    plt.axis('off')
    plt.show()
    
    return {"text": text, "conf": conf}


if front is not None:
    print("Testing all fields...")
    print("=" * 60)
    
    all_results = {}
    for name, f in FRONT_FIELDS.items():
        result = ocr_field(front, f["x"], f["y"], f["w"], f["h"], name, f["lang"])
        if result:
            all_results[name] = result
    
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    for name, r in all_results.items():
        status = ">>" if r["conf"] > 0.5 else "!!"
        print(f"  {status} {name:20s} -> '{r['text']}'")

---
# SECTION 4: COMPARISON

**RAW approach:**
- No calibration needed
- Works on any card
- But no field labels

**Field-based approach:**
- More accurate
- Known field names
- But requires calibration